# Activity 1: Pandas `.plot` Fundamentals

Every DataFrame and Series in pandas has a `.plot` method built on top of matplotlib. You do not need
to import a separate charting library to answer a first question about your data.

In this notebook you will work with two datasets:

- `hospital_claims.parquet`: Medicare hospital billing data, one row per hospital per diagnosis group.
- `closing_price.csv`: daily closing stock prices for AAPL, MSFT, and IBM.

You will build a bar chart from the claims data and a line chart from the price data, adding one
formatting option at a time so you can see what each one does.

Import `pandas` for the data and `matplotlib.pyplot` for the parts of a chart that `.plot` does not
control directly, such as `plt.show()` and `plt.savefig()`. Expect no output, the cell just runs.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

Load the hospital claims data. It is stored as a parquet file, so use `pd.read_parquet`, not
`pd.read_csv`. Expect no visible output yet, this just loads the DataFrame into memory.

In [ ]:
claims = pd.read_parquet("data/hospital_claims.parquet")

Confirm what you loaded before you plot anything. Expect a shape of `(163065, 12)` and a preview
of the columns, including `Provider State`, `Average Covered Charges`, and `Average Total Payments`.

In [ ]:
print(claims.shape)
claims.head(3)

A chart answers a question. The question here: which states bill the most, relative to what
Medicare actually pays them? To answer it you need a ratio, not the raw dollar amounts.

Build a `ratio` column (charges divided by payments) and average it by state. Expect New Jersey,
Nevada, and California at the top, all billing more than 5 times what Medicare pays.

In [ ]:
claims["ratio"] = claims["Average Covered Charges"] / claims["Average Total Payments"]
by_state = claims.groupby("Provider State")["ratio"].mean().sort_values(ascending=False)
by_state.head()

See that aggregate as a chart, with no formatting at all. Expect a bar chart that technically
works but has no title, no axis labels, cramped state codes, and an arbitrary default color.

In [ ]:
by_state.head(15).plot(kind="bar")
plt.show()

The default chart above is unreadable on its own. `.plot` accepts keyword arguments that fix this,
one at a time:

- `title`: what the chart is claiming, ideally the finding, not a generic label.
- `ylabel` and `xlabel`: what each axis measures.
- `figsize` and `rot`: chart size and how far the tick labels rotate.
- `color` and `legend`: the bar color, and whether to show a legend you do not need.
- `grid`: light gridlines to make bar heights easier to compare.

Apply them one at a time, starting with `title`.

In [ ]:
by_state.head(15).plot(
    kind="bar",
    title="New Jersey hospitals bill 6.7 times what Medicare pays",
)
plt.show()

Add `ylabel` and `xlabel` so the reader knows what the bars and the categories represent.

In [ ]:
by_state.head(15).plot(
    kind="bar",
    title="New Jersey hospitals bill 6.7 times what Medicare pays",
    ylabel="Average charge to payment ratio",
    xlabel="State",
)
plt.show()

Add `figsize` to make the chart wider, and `rot` to rotate the state labels so they stop
overlapping.

In [ ]:
by_state.head(15).plot(
    kind="bar",
    title="New Jersey hospitals bill 6.7 times what Medicare pays",
    ylabel="Average charge to payment ratio",
    xlabel="State",
    figsize=(12, 5),
    rot=45,
)
plt.show()

Add a fixed `color` so every bar is deliberate instead of matplotlib's default blue, and turn off
`legend`, a single-series bar chart does not need one.

In [ ]:
by_state.head(15).plot(
    kind="bar",
    title="New Jersey hospitals bill 6.7 times what Medicare pays",
    ylabel="Average charge to payment ratio",
    xlabel="State",
    figsize=(12, 5),
    rot=45,
    color="#B31B1B",
    legend=False,
)
plt.show()

Add `grid` for light horizontal gridlines, and call `plt.tight_layout()` before `plt.show()` so the
rotated labels are not cut off at the edge of the figure. This is the fully formatted chart.

In [ ]:
by_state.head(15).plot(
    kind="bar",
    title="New Jersey hospitals bill 6.7 times what Medicare pays",
    ylabel="Average charge to payment ratio",
    xlabel="State",
    figsize=(12, 5),
    rot=45,
    color="#B31B1B",
    legend=False,
    grid=True,
)
plt.tight_layout()
plt.show()

The same formatting levers apply to a time series. Load the closing price data with
`parse_dates` so the `Date` column becomes real dates, and `index_col` so pandas can plot each
column against the date automatically. Expect a DataFrame indexed by date with one column per
ticker.

In [ ]:
prices = pd.read_csv("data/closing_price.csv", parse_dates=["Date"], index_col="Date")
prices.head(3)

Call `.plot()` on the whole DataFrame with no arguments. Expect one line per column (AAPL, MSFT,
IBM), all sharing the same axes, because pandas uses the index as the x-axis automatically.

In [ ]:
prices.plot(figsize=(12, 5), title="Closing prices, three tickers on one set of axes")
plt.show()

Pass `subplots=True` to give each column its own panel instead of sharing one set of axes. Expect
three stacked charts, one per ticker, each with its own y-axis scale.

In [ ]:
prices.plot(subplots=True, figsize=(12, 6), title="Each ticker on its own panel")
plt.show()

Add `sharex=False` so each panel gets its own independent x-axis instead of one shared axis.
Expect the same three stacked panels, each now drawing its own x-axis ticks and label.

In [ ]:
prices.plot(subplots=True, sharex=False, figsize=(12, 6), title="Each panel with an independent x-axis")
plt.show()

`secondary_y` keeps every column on one chart but moves the named column to a right-hand axis.
Use it when one series would otherwise be squeezed by a different scale. Expect one chart with two
y-axes, IBM's line measured against the right-hand axis.

In [ ]:
prices.plot(secondary_y="IBM", figsize=(12, 5), title="IBM on a secondary axis")
plt.show()

Comparing series on different scales is easier once you normalize them. Divide every row by the
first row so each ticker starts at 1.0. Expect three lines that all begin at the same point and then
diverge based on relative growth, not dollar price.

In [ ]:
normalized = prices.div(prices.iloc[0])
normalized.plot(
    title="Normalized to the first trading day, all three start at 1.0",
    ylabel="Growth relative to day one",
    xlabel="Date",
    figsize=(12, 5),
)
plt.tight_layout()

Save the chart to a file with `plt.savefig`, called before `plt.show()` so the file gets the
finished figure. Expect a PNG file to appear in this notebook's folder, then the same chart
displayed inline.

In [ ]:
plt.savefig("normalized_closing_prices.png")
plt.show()

## Your turn

Three exercises. Reuse the formatting levers from this notebook. Each one should produce a chart
you would be comfortable showing to someone else.

**Exercise 1.** Plot the 15 states with the **lowest** charge-to-payment ratio, formatted to the
same standard as the bar chart above (title, ylabel, xlabel, figsize, rot, color, legend, grid). The
title must state the finding, not just label the axes. Expected: Maryland appears last, at about
1.06.

Maryland's ratio is not noise. Maryland runs an all-payer hospital rate-setting system, the only
one in the United States: a state commission sets the rate every payer (Medicare, Medicaid, and
private insurers) pays for the same hospital service. That is why Maryland's hospitals bill almost
exactly what Medicare pays, while New Jersey's bill 6.7 times as much. A chart that shows an outlier
is not finished until you know why the outlier is there.

**Exercise 2.** Plot the distribution of `Total Discharges` with `kind="hist"` and `bins=50`.
Expected: a hard right skew, most providers under 100 discharges, with a long thin tail of much
larger hospitals.

**Exercise 3.** Plot `Average Covered Charges` against `Average Total Payments` with
`kind="scatter"`. Then, in one sentence, say what the shape of the scatter tells you about the
relationship between the two.